# Tool-RL SFT (Phi-2) - Google Colab

This notebook clones the project repo and runs SFT in Colab using `microsoft/phi-2` as the starting model.
It runs the repo's own code, so push your changes before running it here (set `BRANCH` to test a branch).

In [ ]:
import os

REPO_URL = "https://github.com/codyho22/learn2call.git"
BRANCH = "main"
REPO_DIR = "/content/learn2call"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}/tool-rl-project
!git log --oneline -1

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
%%writefile experiments/config.colab.phi2.yaml
run_name: colab_phi2
seed: 42

model:
  model_id: "microsoft/phi-2"
  trust_remote_code: true
  tokenizer_id: "microsoft/phi-2"

model_name: "microsoft/phi-2"
tokenizer_name: "microsoft/phi-2"

train:
  sft_epochs: 1
  per_device_batch_size: 1
  gradient_accumulation_steps: 8
  gradient_checkpointing: true
  learning_rate: 5.0e-5
  warmup_steps: 100
  max_length: 1024
  fp16: true
  bf16: false

data:
  processed_path: "data/processed/train.jsonl"
  eval_path: "data/processed/val.jsonl"

logging:
  output_dir: "experiments/runs"
  log_interval: 20

In [ ]:
!python src/data.py --splits train --val-ratio 0.05 --seed 42

In [ ]:
!python experiments/test_setup.py

In [ ]:
!python src/train_sft.py --config experiments/config.colab.phi2.yaml --max-train-samples 1000 --max-eval-samples 100 --max-steps 200

In [ ]:
!ls -lah experiments/runs/colab_phi2/final